In [91]:
! pip install -q youtube-transcript-api faiss-cpu tiktoken pytube

In [92]:
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import logging
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import YoutubeLoader
load_dotenv()

True

> ## Step 1a - Indexing (Document Ingestion)

In [93]:
loader = YoutubeLoader.from_youtube_url(
    "https://youtu.be/o6l6tJQgUg4?si=QUTe3v1NO5PZGHE_", add_video_info=False
)
transcript = loader.load()[0].page_content
print(transcript)

Okay, let's do this. This is lecture one of what I'm describing as the RLHF course. I've been working on the RLHF book. Plenty of links will be below. Most people already know this and I hope that the viewership of this will grow over time. I will do accompanying videos with questions and context for this course. Everything will be at rlhfbook.com. This first lecture is an overview of RLHF post-training, how we got where we are with language models, and we'll have a very different tone than most of the lectures at least until maybe later on. This is all about what RLHF is, reinforcement learning from human feedback, how it became post-training, how I view the world, and where the exciting things are now. And you can troll me in the comments for it still being the reinforcement learning from human feedback book. It will be that and as we go through the lectures you will understand why. So let's get into it. To start, this will start broad as broad about as broad audiences as I will get 

> ## Step 1b - Indexing (Text Splitting)

In [94]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [95]:
len(chunks)

59

In [96]:
chunks[0].page_content

"Okay, let's do this. This is lecture one of what I'm describing as the RLHF course. I've been working on the RLHF book. Plenty of links will be below. Most people already know this and I hope that the viewership of this will grow over time. I will do accompanying videos with questions and context for this course. Everything will be at rlhfbook.com. This first lecture is an overview of RLHF post-training, how we got where we are with language models, and we'll have a very different tone than most of the lectures at least until maybe later on. This is all about what RLHF is, reinforcement learning from human feedback, how it became post-training, how I view the world, and where the exciting things are now. And you can troll me in the comments for it still being the reinforcement learning from human feedback book. It will be that and as we go through the lectures you will understand why. So let's get into it. To start, this will start broad as broad about as broad audiences as I will get

> ## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [97]:
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},  # or 'cuda' if GPU available
    encode_kwargs={'normalize_embeddings': False}  # Set to True for cosine similarity
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9816.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [98]:
vectors = embeddings.embed_documents(transcript)


In [99]:
vector_store = FAISS.from_documents(chunks,embeddings)

In [100]:
vector_store.index_to_docstore_id

{0: 'a257eeef-6c5b-4c5d-8ecb-59661f9d1f4c',
 1: 'de4f4b74-c8be-486a-965a-dcfbab4fa1bf',
 2: '8a7bd168-a127-4556-9f82-6e853a8268f6',
 3: '8e7962a9-6783-4d09-9cf6-52afae8adff0',
 4: '9cb938d9-e88d-4158-8557-640e98a31c7a',
 5: '9d804e73-ec16-4825-864a-09ba5f98914f',
 6: '5569f183-f4b7-47d7-af04-ad2cf4ae7a20',
 7: 'af44f19a-eb94-4878-a3fc-d53dccdddc31',
 8: '2931f372-08e5-4cd8-883c-978495556598',
 9: 'd0343d6a-e2d8-4bb2-b02a-0ba91f4bd798',
 10: 'e3cd710b-37da-4c70-b170-f36612360c20',
 11: 'dda6dd12-6143-4aa4-b770-ffe2b254f6ab',
 12: '19a0de58-a28d-492c-b1be-e45e473e7828',
 13: 'd8840f14-3305-4812-8df9-898bff83563b',
 14: '7d4dd7ab-fdca-4508-942f-c97f2e697343',
 15: '55564910-ffcc-4d7c-a180-67327afde193',
 16: 'd2cad06c-b553-47f2-8666-b40651cbbd6c',
 17: '495be438-8a51-42f2-a3a6-4289fb0b5ddc',
 18: 'a83efc2d-a3f8-4ecf-a689-dbe5bd6d844e',
 19: 'ed46ab1f-6e59-4d36-9e47-d3ed0a723dc1',
 20: '16379244-83f8-4866-ba1b-98290e3c2109',
 21: '94cbc81a-24d3-4101-9172-765b162355dd',
 22: '85d86672-2824-

In [101]:
vector_store.get_by_ids(['820e758c-e514-498b-aa67-9041977daad6'])

[]

> ## Step 2 - Retrieval

In [102]:
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={'k':4})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x74ac07fb08f0>, search_kwargs={'k': 4})

In [103]:
retriever.invoke('What is RLHF')

[Document(id='dff4d979-272d-4756-b3ac-5d8fa3ee56d7', metadata={}, page_content="we're starting from a strong foundation of language and we want to keep the strong priors of the language model rather than learning from scratch in traditional RL. So we'll see a lot of these diagrams throughout as we go through. And this is one of the most famous diagrams in RLHF's history, which is like the classic three-step RLHF recipe, which is step one, collect demonstration data and train a supervised policy. Step two, collect comparison data and train a reward model. Step three, do the actual RL, optimize the policy against the reward model using RL using RL. This is all a bit outdated, but it is shows the simplest possible way to get to RLHF. So you need to do instruction tuning and you need to collect data in order to actually do this RL. And this is what um InstructGPT showed. We're going to go into each of the steps. So the step one instruction tuning is really about getting the highest quality

> ## Step 3 - Augmentation

In [104]:
llm = ChatGroq(model='llama-3.3-70b-versatile',temperature=0.2)

In [105]:
prompt = PromptTemplate(
    template = """
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.

        {context}
        Question: {question}
""",
        input_variables=['context','question']
)

In [106]:
question          = "is the topic of large language models discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [107]:
retrieved_docs[0].page_content

"good to note that language models were really fine-tuned for a specific set of tasks and this sort of generalization wasn't assumed to be the case in all of the models. So it's wild to hear today in 2026, but people were very alarmed and totally changed their career trajectories based on the performance of GPT-3. It was very surprising and very unex- it's just unexpected for people. This was still before I was working on language models, so I was out of this loop. In 2021 there was a paper called On the Dangers of Stochastic Parrots: Can Language Models Be Too Big? I put this in to show that there was still a major question over what the value of language models were and how this was going to be scaled. I think some directional arguments of this paper are likely still true which is like what is the cost of building this giant technology. But you can look at it by the name Stochastic Parrots. And there is a population of people that think the language models are just this and they're"

In [108]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"good to note that language models were really fine-tuned for a specific set of tasks and this sort of generalization wasn't assumed to be the case in all of the models. So it's wild to hear today in 2026, but people were very alarmed and totally changed their career trajectories based on the performance of GPT-3. It was very surprising and very unex- it's just unexpected for people. This was still before I was working on language models, so I was out of this loop. In 2021 there was a paper called On the Dangers of Stochastic Parrots: Can Language Models Be Too Big? I put this in to show that there was still a major question over what the value of language models were and how this was going to be scaled. I think some directional arguments of this paper are likely still true which is like what is the cost of building this giant technology. But you can look at it by the name Stochastic Parrots. And there is a population of people that think the language models are just this and they're\n

In [109]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n\n        good to note that language models were really fine-tuned for a specific set of tasks and this sort of generalization wasn't assumed to be the case in all of the models. So it's wild to hear today in 2026, but people were very alarmed and totally changed their career trajectories based on the performance of GPT-3. It was very surprising and very unex- it's just unexpected for people. This was still before I was working on language models, so I was out of this loop. In 2021 there was a paper called On the Dangers of Stochastic Parrots: Can Language Models Be Too Big? I put this in to show that there was still a major question over what the value of language models were and how this was going to be scaled. I think some directional arguments of this paper are likely still true which is like wha

> ## Step 4 - Generation

In [110]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of large language models is discussed in this video. 

The discussion includes:
- The surprise and alarm caused by the performance of GPT-3, which led some people to change their career trajectories.
- The question of the value and scalability of language models, as raised in a 2021 paper called "On the Dangers of Stochastic Parrots: Can Language Models Be Too Big?"
- The evolution of language models, including the transformer architecture and its use of self-attention mechanisms.
- The growth of language models in terms of parameters, with the best ones now having billions to trillions of parameters.
- The application of language models to multimodal projects, including images, audio, and video.
- The definition and basics of language models, including their use of probabilities and auto-regressive nature to predict and generate text.


> ## Building Chain

In [111]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [112]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [113]:
parallel_chain = RunnableParallel(
    {
        "context" : retriever | RunnableLambda(format_docs),
        'question' : RunnablePassthrough()
    }
)

In [114]:
parallel_chain.invoke('What is RLHF')

{'context': "we're starting from a strong foundation of language and we want to keep the strong priors of the language model rather than learning from scratch in traditional RL. So we'll see a lot of these diagrams throughout as we go through. And this is one of the most famous diagrams in RLHF's history, which is like the classic three-step RLHF recipe, which is step one, collect demonstration data and train a supervised policy. Step two, collect comparison data and train a reward model. Step three, do the actual RL, optimize the policy against the reward model using RL using RL. This is all a bit outdated, but it is shows the simplest possible way to get to RLHF. So you need to do instruction tuning and you need to collect data in order to actually do this RL. And this is what um InstructGPT showed. We're going to go into each of the steps. So the step one instruction tuning is really about getting the highest quality completion data. Back in InstructGPT's time, all the completions w

In [115]:
parser = StrOutputParser()

In [116]:
main_chain = parallel_chain | prompt | llm | parser
main_chain.invoke('Can you summarize the content')

'The content appears to be a lecture or discussion about language models, specifically post-training and the process of improving their performance. The speaker mentions the importance of understanding the foundations of language models, including how they work and the role of post-training in achieving better results. They also touch on the idea that the base model determines the ceiling of performance, but post-training is necessary to reach that ceiling. The speaker is working on a book and invites listeners to get in touch with them on social media or GitHub for issues or questions.'